# this needs to be written from-scratch


3 sources of gene information:

```
disease-gene.csv
"source","source_type","target","target_type"
"DOID_7551","disease","ENSG00000058085","gene"

drug-gene.csv
"source","source_type","target","target_type"
"CHEMBL1201746","drug","ENSG00000228716","gene"

gene-reactome.csv
"source","source_type","target","target_type"
"ENSG00000004948","gene","R-HSA-418555","reactome"
```


In [1]:
require 'csv'

allgenes = {}

['./rawdata/disease-gene.csv', './rawdata/drug-gene.csv'].each do |file|  
  CSV.foreach(file, col_sep: ",", quote_char: '"', 
  liberal_parsing: true, headers: true) do |row|
    gene_id = row['target']
    allgenes[gene_id] = 1
  end
end

['./rawdata/gene-reactome.csv'].each do |file|  
  CSV.foreach(file, col_sep: ",", quote_char: '"', 
  liberal_parsing: true, headers: true) do |row|
    gene_id = row['source']
    allgenes[gene_id] = 1
  end
end

# allgenes now contains all of the genes as hash keys

["./rawdata/gene-reactome.csv"]

In [3]:
allgenes

{"ENSG00000058085"=>1, "ENSG00000091831"=>1, "ENSG00000102755"=>1, "ENSG00000105329"=>1, "ENSG00000112115"=>1, "ENSG00000119888"=>1, "ENSG00000012779"=>1, "ENSG00000068024"=>1, "ENSG00000082701"=>1, "ENSG00000092969"=>1, "ENSG00000100387"=>1, "ENSG00000101076"=>1, "ENSG00000133742"=>1, "ENSG00000146648"=>1, "ENSG00000164692"=>1, "ENSG00000171105"=>1, "ENSG00000176890"=>1, "ENSG00000182578"=>1, "ENSG00000232810"=>1, "ENSG00000073756"=>1, "ENSG00000115594"=>1, "ENSG00000134318"=>1, "ENSG00000134871"=>1, "ENSG00000143799"=>1, "ENSG00000160712"=>1, "ENSG00000163558"=>1, "ENSG00000169083"=>1, "ENSG00000177455"=>1, "ENSG00000189221"=>1, "ENSG00000196411"=>1, "ENSG00000197122"=>1, "ENSG00000067900"=>1, "ENSG00000105675"=>1, "ENSG00000110799"=>1, "ENSG00000112715"=>1, "ENSG00000120907"=>1, "ENSG00000157184"=>1, "ENSG00000197565"=>1, "ENSG00000254087"=>1, "ENSG00000077514"=>1, "ENSG00000077782"=>1, "ENSG00000082898"=>1, "ENSG00000108821"=>1, "ENSG00000108839"=>1, "ENSG00000132170"=>1, "ENSG0000

In [2]:
genequery = "
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX uniprotkb: <http://purl.uniprot.org/uniprot/>
PREFIX taxon: <http://purl.uniprot.org/taxonomy/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX up: <http://purl.uniprot.org/core/>

SELECT distinct ?ensid ?geneid ?protein ?recommended_full ?taxon
WHERE
{
        VALUES ?ensid {|||VALUES|||}
        ?protein a up:Protein .
  		?protein rdfs:seeAlso ?ensid . # seeAlso http://purl.uniprot.org/opentargets/ENSG00000176014
  		?protein rdfs:seeAlso ?geneid . # seeAlso http://purl.uniprot.org/geneid/939976
  		?protein up:organism ?taxon .
  		?protein up:recommendedName ?rname .
  		?rname up:fullName ?recommended_full .
        FILTER(CONTAINS(str(?geneid), 'purl.uniprot.org/geneid'))
}"
puts


In [3]:
def format_ensembl_values_clause(idlist:, batch_size: 30)
  # used to make efficient sparql
  valueslist = Array.new
  puts idlist.size
#   slice = 1
  base_uri = "<http://purl.uniprot.org/opentargets/%s>"
  idlist.each_slice(batch_size).map do |batch|
#     puts slice
#     slice = slice + 1
    values = batch.map { |id| base_uri % id }.join(' ')
    valueslist << values
  end
  valueslist  # valueslist is a list of lists of 20 elements
end

:format_ensembl_values_clause

# need this output structure

sourceid,label,geneid,protein,recommended_full,taxon
ENSG00000091831,ENSG00000091831,http://purl.uniprot.org/geneid/2099,http://purl.uniprot.org/uniprot/P03372,Estrogen receptor,http://purl.uniprot.org/taxonomy/9606

In [ ]:
require 'sparql/client'

out = File.open("./maps/genes.map", "w");
out.write "sourceid,label,geneid,protein,recommended_full,taxon\n"
puts "START"

sparql = SPARQL::Client.new("https://sparql.uniprot.org/sparql/")

batches = format_ensembl_values_clause(idlist: allgenes.keys)
batches.each do |batch|
  retry_attempts = 0
  begin
    result = sparql.query(genequery.gsub("|||VALUES|||", batch))
  rescue StandardError => e
    retry_attempts += 1
    if retry_attempts < 10
      warn "retrying"
      retry
    else
      warn e.inspect
      abort
    end
  end
  puts "FOUND: #{result.size}"
#  abort "#{batch}\n#{result.inspect}" unless result.size >= 30
  result.each do |res|
    # SELECT ?ensid ?geneid ?protein ?recommended_full ?taxon
    ensuri = res["ensid"].to_s
    if match = ensuri.match(/.*\/([\w\d]+)/)  # get rid of the URI part to go back to source formatting
      ensid = match[1]
      label = res["recommended_full"].to_s
    else
      abort "no ensuri match #{ensuri}"
    end
    geneuri = res["geneid"].to_s
    
    #"sourceid,label,geneid,protein,recommended_full,taxon\n"
    puts "#{ensid},#{label},#{geneuri},#{res['protein']},#{res['recommended_full']},#{res['taxon']}"
    out.write CSV.generate_line([ensid,label,geneuri,res["protein"],res["recommended_full"],res["taxon"]])
  end
end

out.close
puts "done ensembl genes"